# ReAct (Reasoning + Acting)

**Domain:** Agentic AI  ·  **runnable:** yes

A refresher on **ReAct** — the *Reasoning + Acting* prompting pattern from Yao et al. (2022, "ReAct: Synergizing Reasoning and Acting in Language Models"). ReAct is the loop almost every modern tool-using agent runs underneath: the model interleaves **Thought** (free-text reasoning), **Action** (a tool call), and **Observation** (the tool's result), repeating until it emits a **Final Answer**.

ReAct isn't a library — it's a *technique*. Frameworks like [[langchain]], [[langgraph]], [[crewai]], and [[smolagents]] all implement some flavour of it. Understanding the bare loop makes those frameworks legible, and lets you build a capable single-file agent in ~60 lines.

## 1. What & Why

A plain chain-of-thought prompt makes the model *reason* ("think step by step") but it reasons in a vacuum: it can't look anything up, run code, or check a fact, so it hallucinates and can't recover. A plain tool-calling prompt lets the model *act* but with no scratchpad to plan, decide *why* a tool is needed, or react to a surprising result.

**ReAct fuses the two.** At each step the model first writes a *Thought* (what do I know, what do I need next), then an *Action* (call a tool with an argument). Your code runs the tool and feeds the result back as an *Observation*. The model reads it, thinks again, and either acts once more or stops with a Final Answer.

**The problem it solves.** This closed loop gives the model three things a one-shot prompt can't:

- **Grounding** — facts come from tools (search, a database, a calculator, an API), not the model's memory, so answers are checkable and current.
- **Recovery** — a bad Observation (empty result, error, wrong page) is just more context; the model can adjust its next Thought and try a different action instead of committing to a wrong path.
- **Decomposition** — multi-hop questions ("the population of the country whose capital is Canberra") get broken into a sequence of grounded sub-steps.

**When to reach for it.** Any task that needs *external* information or side effects to answer correctly, where the steps aren't known in advance. **When not to:** if the task is pure reasoning with no tools (just use chain-of-thought), or if the workflow is fixed and known ahead of time (a hard-coded pipeline is cheaper, faster, and more reliable than letting the model decide each step).

## 2. Mental Model

**ReAct is a REPL where the model is the programmer and your tools are the standard library.**

At a read-eval-print loop, you type an expression, the interpreter evaluates it and prints a result, you read the result and type the next expression. ReAct is the same loop with the roles shifted:

```
   ┌─────────────────────────────────────────────┐
   │  Thought:  reason about what to do next       │  ← model writes
   │  Action:   tool_name[input]                    │  ← model writes
   │  Observation: <result of running the tool>     │  ← YOUR CODE writes
   └───────────────────────┬─────────────────────┘
                           │  (append to scratchpad, loop)
                           ▼
            ... repeat until ...
   │  Thought:  I now know the answer               │
   │  Final Answer: <the answer>                    │  ← model writes, loop ends
```

The **scratchpad** is the whole conversation so far — every Thought/Action/Observation, concatenated. Each turn you send the prompt + scratchpad back to the model and it generates the *next* Thought + Action. The single most important implementation trick: **stop generation right after the Action**, before the model hallucinates its own Observation. Your code produces the real Observation; you never let the model invent it.

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **Thought** | The model's free-text reasoning for the current step — its plan for *why* it's about to act. Pure chain-of-thought, not parsed by your code. |
| **Action** | A structured tool call the model emits, classically as `tool_name[argument]`. Your code parses this and dispatches to the matching tool. |
| **Observation** | The tool's return value, written back into the scratchpad by **your code** (never the model). Becomes context for the next Thought. |
| **Scratchpad / trajectory** | The growing transcript of all prior Thought/Action/Observation steps, re-sent every turn. This is the agent's working memory. |
| **Tool / action space** | The finite set of tools the agent may call. Described in the system prompt (name, what it does, argument format). Small, well-described tool sets work far better than large vague ones. |
| **Stop sequence** | The string (e.g. `"Observation:"`) that halts model generation after it writes an Action, so it can't fabricate the result. The mechanical heart of ReAct. |
| **Final Answer** | The terminal action. When the model emits it, the loop ends and you return it to the user. |
| **Max steps** | A hard cap on loop iterations. Without it a confused agent loops forever (and bills forever). Non-negotiable in production. |

## 4. Setup

Examples 1 and 2 are **pure standard library** — they run with no install and no API key, using a deterministic scripted "model" so the loop is reproducible and offline. Example 3 calls a real LLM and is gated behind an API-key check.

```bash
# Only needed for Example 3 (the live-LLM cell):
pip install anthropic
export ANTHROPIC_API_KEY=sk-ant-...
```

Requires **Python 3.9+**. The cell below just reports what's available; nothing is required for the core examples.

In [ ]:
# Environment check — Examples 1 & 2 need nothing; Example 3 needs a key + the SDK.
import importlib.util, os

has_anthropic = importlib.util.find_spec("anthropic") is not None
has_key = bool(os.getenv("ANTHROPIC_API_KEY"))

print("anthropic SDK installed:", has_anthropic)
print("ANTHROPIC_API_KEY present:", has_key)
print()
print("Examples 1 & 2 run regardless (stdlib only, offline, deterministic).")
print("Example 3 runs only if both of the above are True.")

## 5. Worked Examples

### Example 1 — A ReAct engine from scratch

Below is a complete ReAct loop in ~40 lines. The pieces are exactly what a framework like LangChain wires up for you:

1. a **tool registry** (here: a `calc` calculator and a tiny `wiki` knowledge base),
2. a **parser** that pulls `Action: tool[input]` / `Final Answer: ...` out of model text,
3. the **loop** that runs the tool, appends the Observation, and re-prompts — capped by `max_steps`.

The "model" is a deterministic `scripted_llm` so the notebook runs offline and reproducibly. It reads the current scratchpad and returns the next Thought + Action — exactly the text a real LLM would generate. (Example 3 swaps it for a live model with no other changes.)

In [ ]:
import re

# --- 1. Tools: each is a plain function tool(arg:str) -> str -------------------
WIKI = {
    "france": "France is a country in Western Europe. Its capital is Paris.",
    "paris": "Paris is the capital of France. Population: 2,100,000.",
}

def wiki(query: str) -> str:
    return WIKI.get(query.strip().lower(), f"No article found for {query!r}.")

def calc(expr: str) -> str:
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))   # toy; never eval untrusted input
    except Exception as e:
        return f"Error: {e}"

TOOLS = {"wiki": wiki, "calc": calc}

# --- 2. Parser: find the next Action or a Final Answer in model output ---------
ACTION_RE = re.compile(r"Action:\s*(\w+)\[(.*?)\]", re.DOTALL)
FINAL_RE  = re.compile(r"Final Answer:\s*(.*)", re.DOTALL)

def parse(text: str):
    if (m := FINAL_RE.search(text)):
        return ("final", m.group(1).strip())
    if (m := ACTION_RE.search(text)):
        return ("action", (m.group(1), m.group(2)))
    return ("none", text)

# --- 3. The ReAct loop --------------------------------------------------------
def react(question, llm, max_steps=6, verbose=True):
    scratchpad = f"Question: {question}\n"
    for step in range(1, max_steps + 1):
        thought_action = llm(scratchpad)          # model writes Thought + Action
        scratchpad += thought_action
        kind, payload = parse(thought_action)
        if verbose:
            print(thought_action.strip())
        if kind == "final":
            return payload
        if kind == "action":
            tool, arg = payload
            obs = TOOLS[tool](arg) if tool in TOOLS else f"Unknown tool {tool!r}"
            scratchpad += f"\nObservation: {obs}\n"
            if verbose:
                print(f"Observation: {obs}\n")
    return "Stopped: hit max_steps without a Final Answer."

In [ ]:
# A deterministic stand-in for a real LLM. It inspects the scratchpad and returns
# the next Thought + Action, mimicking exactly what a trained model would emit.
def scripted_llm(scratchpad: str) -> str:
    if "Observation:" not in scratchpad:                       # step 1: look it up
        return ("Thought: I should look up the population of Paris.\n"
                "Action: wiki[Paris]")
    if "Population: 2,100,000" in scratchpad and "Final Answer" not in scratchpad:
        return ("Thought: The article gives the population directly.\n"
                "Final Answer: Paris has a population of about 2.1 million.")
    return "Final Answer: (unsure)"

answer = react("What is the population of Paris?", scripted_llm)
print("=" * 40)
print("RETURNED:", answer)

### Example 2 — Multi-hop reasoning + a tool chain

The same engine, unchanged, now handles a question that **no single tool call can answer**: *"What is double the population of the capital of France?"* The agent must (1) find the capital, (2) look up its population, (3) do arithmetic — interleaving two different tools and carrying intermediate results forward in the scratchpad. This decomposition is the whole point of ReAct: the trajectory is discovered step by step, not planned up front.

In [ ]:
def scripted_llm_multihop(scratchpad: str) -> str:
    obs_count = scratchpad.count("Observation:")
    if obs_count == 0:                                   # step 1: which capital?
        return ("Thought: First I need the capital of France.\n"
                "Action: wiki[France]")
    if obs_count == 1:                                   # step 2: its population
        return ("Thought: The capital is Paris. Now I need its population.\n"
                "Action: wiki[Paris]")
    if obs_count == 2:                                   # step 3: arithmetic
        return ("Thought: Population is 2,100,000. I must double it.\n"
                "Action: calc[2100000 * 2]")
    return ("Thought: I have the doubled figure.\n"
            "Final Answer: Double the population of Paris (France's capital) is 4,200,000.")

answer = react("What is double the population of the capital of France?",
               scripted_llm_multihop)
print("=" * 40)
print("RETURNED:", answer)

### Example 3 — The same loop with a real LLM (gated)

Swapping the scripted model for a real one changes **one function**: the `llm` callback. With a live model the only new ingredient is the **stop sequence** — you tell the API to stop generating at `"Observation:"` so the model writes a Thought and an Action and then *halts*, letting your code produce the real Observation. This is the authentic ReAct API pattern (the same trick LangChain's ReAct agent uses).

This cell runs only if `anthropic` is installed and `ANTHROPIC_API_KEY` is set; otherwise it prints the call shape and skips, so the notebook still executes top-to-bottom.

In [ ]:
SYSTEM = """Answer the question by reasoning and acting in a loop. Each turn, output:
Thought: <your reasoning>
Action: <tool>[<input>]
where <tool> is one of: wiki[query], calc[python_expression].
After you see an Observation, continue. When done, output:
Final Answer: <answer>"""

def make_anthropic_llm():
    import anthropic
    client = anthropic.Anthropic()                      # reads ANTHROPIC_API_KEY

    def llm(scratchpad: str) -> str:
        msg = client.messages.create(
            model="claude-haiku-4-5",                   # small + cheap is plenty for ReAct
            max_tokens=256,
            system=SYSTEM,
            messages=[{"role": "user", "content": scratchpad}],
            stop_sequences=["Observation:"],            # <-- the heart of ReAct
        )
        return msg.content[0].text
    return llm

if has_anthropic and has_key:
    live_llm = make_anthropic_llm()
    print(react("What is the population of the capital of France?", live_llm))
else:
    print("Skipping live call (no SDK or no ANTHROPIC_API_KEY).")
    print("With a key set, react(...) would drive a real Claude model through the")
    print("exact same Thought/Action/Observation loop as Examples 1 and 2.")

## 6. Gotchas & Pitfalls

- **Letting the model write its own Observation.** Without a stop sequence (or equivalent), the model happily continues past `Action:` and *hallucinates* a plausible-looking Observation — defeating the entire point of grounding. Always stop generation at `Observation:` and inject the real tool result yourself.
- **No step cap = infinite loop.** A confused agent will Thought→Action→Thought→Action forever, burning tokens and money. `max_steps` (plus ideally a token/cost budget) is mandatory, not optional.
- **Brittle action parsing.** Real models drift from the format — extra prose, markdown fences, `Action: wiki("Paris")` instead of `wiki[Paris]`, multiple actions at once. Make the parser tolerant, and on a parse failure feed an Observation like *"Couldn't parse your action; use the format tool[input]"* so the model self-corrects instead of crashing.
- **Vague or too-many tools.** The model picks tools from their descriptions. Sprawling tool sets with fuzzy descriptions cause wrong-tool / wrong-argument errors. Keep the action space small and each tool's purpose and argument format crisp. Prefer **native tool calling / function calling** for production — modern models emit structured tool calls reliably, which sidesteps text parsing entirely (ReAct-the-prompt becomes ReAct-the-loop over a function-calling API).
- **Scratchpad blowup.** The full trajectory is re-sent every step, so long runs grow context (and cost) quadratically and eventually overflow the window. Summarize or trim old steps, or cap the horizon.
- **Treating Observations as ground truth.** Tools can return errors, empty results, or wrong pages. A good ReAct prompt encourages the model to notice a bad Observation and try a different action rather than charging ahead. Garbage Observations become poisoned context if unhandled.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-offs vs ReAct |
|---|---|---|
| **ReAct (this)** | Tasks needing external info/tools where the steps aren't known in advance; the canonical single-agent tool loop | You own the loop, parsing, stopping, and error handling — flexible but it's on you |
| **Plain chain-of-thought** | Pure reasoning with no external facts or side effects (math, logic, rewriting) | No tools, no grounding — but simpler and one call; use it when no tool is needed |
| **Native tool / function calling** | Production tool use — the model emits *structured* calls instead of text to parse | Same loop, far more robust; the modern default. ReAct's text format is the pedagogical / no-function-calling fallback |
| **Plan-and-Execute / ReWOO** | Tasks where you'd rather plan all steps up front, then execute (fewer LLM calls) | Less adaptive mid-run than ReAct's step-by-step replanning, but cheaper and more parallelizable |
| **[[langgraph]]** | Production agents needing explicit state, branching, persistence, human-in-the-loop | More to wire up; gives you durable, inspectable control over the ReAct-style loop |
| **[[crewai]] / [[autogen]]** | Multi-agent role/conversation workflows | Coordinate several agents; each agent often runs a ReAct loop internally |

**Rule of thumb:** ReAct is the *mechanism*; everything to the right is a way to make that mechanism more robust, more structured, or multi-agent. Learn the bare loop first — it makes every framework above legible.

## 8. Resources

- **ReAct paper — Yao et al., "ReAct: Synergizing Reasoning and Acting in Language Models" (ICLR 2023)** — https://arxiv.org/abs/2210.03629
- **Project page with the original prompts and traces** — https://react-lm.github.io/
- **LangChain ReAct agent (a production implementation of this loop)** — https://python.langchain.com/docs/how_to/agent_executor/
- **Anthropic — Tool use (native function calling, the robust successor to text-parsed ReAct)** — https://docs.anthropic.com/en/docs/build-with-claude/tool-use
- **Lilian Weng, "LLM Powered Autonomous Agents" (ReAct in context of agent design)** — https://lilianweng.github.io/posts/2023-06-23-agent/
- **Prompt Engineering Guide — ReAct** — https://www.promptingguide.ai/techniques/react